In [1]:
from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import xarray as xr

from neuralhydrology.datautils.utils import load_basin_file
from neuralhydrology.datautils.climateindices import calculate_dyn_climate_indices
from neuralhydrology.datautils.pet import get_priestley_taylor_pet

project_root = Path('/publicwork/gauch/GRIP-GL/scripts/MachineLearning')
data_root = Path('/publicdata/Hydrology/')

In [2]:
static_attributes = pd.read_csv(project_root / 'attributes' / 'static_attributes.csv', index_col=[0])

In [3]:
grip_discharge_obj1 = xr.open_dataset(project_root / f'../../data/objective_1/great-lakes/calibration/netcdf/all_gauges.nc')
grip_discharge_obj2 = xr.open_dataset(project_root / f'../../data/objective_2/great-lakes/calibration/netcdf/all_gauges.nc')

def load_grip_discharge(basin: str) -> pd.Series:
    if basin in grip_discharge_obj1['station_id']:
        xarray = grip_discharge_obj1
    elif basin in grip_discharge_obj2['station_id']:
        xarray = grip_discharge_obj2
    else:
        return None
    nstation = np.where(xarray['station_id'].values == basin)[0][0]
    discharge = xarray.sel(nstations=nstation)['Q'].to_series()
    area = static_attributes.loc[basin, 'area_km2'] * 1000_000  # km2 to m2
    discharge.loc[discharge < 0] = np.nan
    discharge.index.name = 'date'
    
    return discharge * (60 * 60 * 24) * 1000.0 / area  # m3/s to mm/day

In [4]:
clim_indices = {}

no_forcing_basins = []
no_discharge_basins  = []
for basin in tqdm(sorted(static_attributes.index)):

    is_spatial_val = False
    basin_rdrsv2_file = project_root / '../../data/ml-met-calibration' / f'ml_met_{basin}.csv'
    val_basin_rdrsv2_file = project_root / '../../data/ml-met-validation-temporal' / f'ml_met_{basin}.csv'
    if not basin_rdrsv2_file.exists():
        is_spatial_val = True
        basin_rdrsv2_file = project_root / '../../data/ml-met-validation-spatial' / f'ml_met_{basin}.csv'
        if not basin_rdrsv2_file.exists():
            no_forcings_basins.append(basin)
            continue
        
    # The aggregated RDRS forcings are already in local standard time (UTC-5)
    if not is_spatial_val:
        # don't use validation data to calculate climate indices
        climidx_forcings = pd.read_csv(basin_rdrsv2_file, index_col=0, parse_dates=[0], skipinitialspace=True)
        climidx_forcings.index.name = 'date'
        # temporal validation forcings also contain the calibration date range
        forcings = pd.read_csv(val_basin_rdrsv2_file, index_col=0, parse_dates=[0], skipinitialspace=True)
        forcings.index.name = 'date'
    else:
        forcings = pd.read_csv(basin_rdrsv2_file, index_col=0, parse_dates=[0], skipinitialspace=True)
        forcings.index.name = 'date'
        climidx_forcings = forcings
        climidx_forcings = climidx_forcings.loc[:'2011-01-01 07:00:00']

    assert np.all(climidx_forcings.index[[0,-1]] == pd.DatetimeIndex(['2000-01-01 08:00:00', '2011-01-01 07:00:00']))

    if forcings.isna().all(axis=None):
        no_forcing_basins.append(basin)
        continue

    lat = static_attributes.loc[basin, 'gauge_lat']

    # resample to daily values: sum(precip), min/max(temp), mean for all other variables
    # run once with calibration data (for climate index calculation), once with full data (for actual forcings)
    for i, forcing_set in enumerate([climidx_forcings, forcings]):
        daily_resampled = forcing_set.resample('1D')
        daily_forcings = daily_resampled.mean()

        # precip
        daily_forcings['RDRS_v2_A_PR0_SFC_m'] = daily_resampled['RDRS_v2_A_PR0_SFC_m'].sum(min_count=1)
        # temp
        daily_forcings['min_RDRS_v2_P_TT_1.5m_degC'] = daily_resampled['RDRS_v2_P_TT_1.5m_degC'].min()
        daily_forcings['max_RDRS_v2_P_TT_1.5m_degC'] = daily_resampled['RDRS_v2_P_TT_1.5m_degC'].max()
        daily_forcings['potential_evapotranspiration'] = \
            get_priestley_taylor_pet(daily_forcings['min_RDRS_v2_P_TT_1.5m_degC'].values,
                                     daily_forcings['max_RDRS_v2_P_TT_1.5m_degC'].values,
                                     daily_forcings['RDRS_v2_P_FB_SFC_W_m2'].values,  # shortwave radiation
                                     lat=lat,
                                     elev=static_attributes.loc[basin, 'mean_elev'],
                                     doy=daily_forcings.index.dayofyear.values)

        if i == 0:
            # since window_length is length of forcings, there will only be one date returned, so we can do .iloc[0]
            clim_indices[basin] = calculate_dyn_climate_indices(daily_forcings['RDRS_v2_A_PR0_SFC_m'] * 1000,  # m to mm
                                                                   daily_forcings['max_RDRS_v2_P_TT_1.5m_degC'],
                                                                   daily_forcings['min_RDRS_v2_P_TT_1.5m_degC'],
                                                                   daily_forcings['potential_evapotranspiration'],
                                                                   window_length=len(daily_forcings)).iloc[0]

    # add discharge
    discharge = load_grip_discharge(basin)
    daily_forcings['qobs_mm_per_day'] = discharge if discharge is not None else np.nan
    if discharge is None:
        no_discharge_basins.append(basin)

    xr_forcings_and_discharge = xr.Dataset.from_dataframe(daily_forcings)
    xr_forcings_and_discharge.to_netcdf(project_root / 'time_series' / f'{basin}.nc')

print(f'No forcings for basins:  ({len(no_forcing_basins)}) {no_forcing_basins}')
print(f'No discharge for basins: ({len(no_discharge_basins)}) {no_discharge_basins}')


No forcings for basins:  (0) []
No discharge for basins: (71) ['02AB006', '02AB021', '02AD012', '02AE001', '02BA006', '02BC004', '02BC006', '02BD007', '02BE002', '02BF014', '02CA007', '02CE002', '02CF010', '02CF011', '02EA018', '02EC011', '02EC019', '02ED027', '02ED032', '02FA004', '02FC016', '02FE008', '02FE010', '02FE013', '02FF002', '02GA047', '02GB007', '02GC010', '02GC026', '02GE007', '02GG013', '02HB011', '02HB029', '02HC025', '02HF002', '02HJ003', '02HL008', '02HM010', '02JB013', '02JE027', '02KC018', '02KD002', '02KF005', '02LB007', '02LB032', '02LC008', '02LC029', '02LD005', '02LE024', '02LE025', '02LG005', '04015330', '04041500', '04057510', '04069500', '04084500', '04087120', '04096015', '04102776', '04108660', '04125550', '04133501', '04159900', '04168400', '04195500', '04201500', '04209000', '04214500', '04220045', '04249000', '04250750']


In [5]:
indices = pd.DataFrame(clim_indices).T
indices.index.set_names('basin', inplace=True)
indices.columns = [col.split('_dyn')[0] for col in indices.columns]
indices.to_csv(project_root / 'attributes/climate_indices.csv')

In [6]:
indices

,p_mean,pet_mean,aridity,t_mean,frac_snow,high_prec_freq,high_prec_dur,low_prec_freq,low_prec_dur
basin,,,,,,,,,
02AB006,2.138044,0.829657,0.388045,2.502294,0.232226,0.046529,1.087209,0.663349,3.835971
02AB017,2.107375,0.847062,0.401951,2.760651,0.250879,0.052252,1.105263,0.683503,4.045655
02AB021,2.081907,0.830730,0.399023,2.176183,0.260394,0.051754,1.136612,0.677532,3.929293
02AC001,2.101688,0.813805,0.387215,2.508344,0.257958,0.048768,1.126437,0.675790,3.852482
02AC002,2.135647,0.805279,0.377066,2.431436,0.250319,0.043792,1.135484,0.663847,3.752461
...,...,...,...,...,...,...,...,...,...
04249000,2.966777,0.995834,0.335662,8.514763,0.178789,0.041055,1.092715,0.548893,2.879896
04250200,4.318828,0.960209,0.222331,6.713038,0.267336,0.044041,1.066265,0.505598,2.768392
04250750,3.731520,0.966845,0.259102,7.159472,0.256835,0.043792,1.041420,0.537447,2.838371
